# ETF MM Arbitrage Simulator — Default Backtest

Reference notebook driving `configs/default.yaml`. Loads the config, runs a reduced-path Monte Carlo sweep, aggregates per-cell analytics, renders the summary table, and produces the per-regime terminal-P&L histograms plus one sample-path diagnostic.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import matplotlib
matplotlib.use('Agg')
from etf_mm_sim.config import load_config, MCParams
from etf_mm_sim.backtest import run_backtest
from etf_mm_sim import analytics, viz
from etf_mm_sim.seeding import analytics_seed
from dataclasses import replace


In [ ]:
cfg = load_config('../configs/default.yaml')
# Reduce n_paths so the notebook executes quickly in CI.
cfg = replace(cfg, mc=MCParams(n_paths=100))


In [ ]:
result = run_backtest(cfg)


In [ ]:
cells = [
    analytics.aggregate_cell(
        result.paths[(s, r.name)],
        s,
        r.name,
        cfg.analytics.sharpe_annualization_factor,
        cfg.analytics.adverse_selection_horizon_steps,
    )
    for s in ('avellaneda_stoikov', 'symmetric', 'semi_as')
    for r in cfg.mid_price.regimes
]
summary = viz.render_summary_table(cells)
summary


In [ ]:
for r in cfg.mid_price.regimes:
    fig = viz.plot_terminal_pnl_hist(result, r.name)


In [ ]:
sample = result.paths[('avellaneda_stoikov', cfg.mid_price.regimes[1].name)][0]
fig = viz.plot_sample_path(sample)
